<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione4/Lezione4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 4 — RAG: Conoscenza Personalizzata

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 28/05/2026

---

### 🎯 Obiettivi
- ✅ Capire la pipeline RAG completa
- ✅ Indicizzare un PDF con ChromaDB
- ✅ Implementare la ricerca semantica
- ✅ Integrare RAG nel chatbot esistente

In [ ]:
# Setup
!pip install anthropic chromadb pypdf sentence-transformers -q
from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

def chiedi_claude(domanda, system=None, max_tokens=800):
    params = {"model":"claude-haiku-4-5-20251001","max_tokens":max_tokens,
              "messages":[{"role":"user","content":domanda}]}
    if system: params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

---
## 1. Crea un documento di test

Per l'esercizio creiamo un documento di testo su WiData. In un progetto reale useresti un PDF vero.

In [ ]:
# Creiamo un documento di testo su WiData
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

# Salva su file
with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    f.write(documento_widata)

print(f"✅ Documento creato: {len(documento_widata)} caratteri")

✅ Documento creato: 1846 caratteri


---
## 2. Chunking e Indicizzazione

In [ ]:
def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunka_testo(documento_widata)
print(f"📊 Numero di chunk: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} char) ---")
    print(chunk[:100]+"...")
    print()

📊 Numero di chunk: 6

--- Chunk 1 (400 char) ---

WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è proge...

--- Chunk 2 (400 char) ---
. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n...

--- Chunk 3 (400 char) ---
km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G ...

--- Chunk 4 (400 char) ---
iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-tim...

--- Chunk 5 (400 char) ---
va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiest...

--- Chunk 6 (96 char) ---
: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...



In [ ]:
import chromadb

# Crea il client ChromaDB in memoria
chroma_client = chromadb.Client()

# Crea o recupera la collection
collection = chroma_client.get_or_create_collection(
    name="widata_docs",
    metadata={"hnsw:space": "cosine"}  # usa similarità coseno
)

# Indicizza i chunk
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")
print("💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 35.0MiB/s]


✅ Indicizzati 6 chunk in ChromaDB
💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!


---
## 3. Ricerca Semantica

In [ ]:
def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Test ricerca semantica
domande_test = [
    "Quali sensori supportate per ambienti esterni?",
    "Come posso integrare i dati con il mio sistema ERP?",
    "Qual è il costo del piano professionale?",
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk[:120]}...")


❓ Quali sensori supportate per ambienti esterni?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Come posso integrare i dati con il mio sistema ERP?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Qual è il costo del piano professionale?
  📄 Chunk 1: : Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...
  📄 Chunk 2: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...


---
## 4. RAG Completo — Domanda + Contesto + Risposta

In [ ]:
SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""

def chat_rag(domanda, n_chunks=3):
    """Chatbot con RAG: recupera contesto e genera risposta."""
    # 1. Recupera i chunk rilevanti
    chunks_rilevanti = cerca(domanda, n_risultati=n_chunks)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    # 2. Costruisci il prompt aumentato
    prompt = f"""Documenti di riferimento:

{contesto}

---

Domanda dell'utente: {domanda}"""

    # 3. Genera la risposta
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    return risposta, chunks_rilevanti

# Test completo
domanda = "Il sensore XS200 funziona in ambienti molto freddi?"
risposta, chunks = chat_rag(domanda)

print(f"❓ {domanda}")
print(f"\n🤖 {risposta}")
print(f"\n📄 Basato su {len(chunks)} chunk")

❓ Il sensore XS200 funziona in ambienti molto freddi?

🤖 # Sensore XS200 in Ambienti Freddi

Sì, il sensore XS200 **funziona in ambienti freddi**. 

Secondo le specifiche tecniche, è in grado di misurare temperature da **-20°C a +60°C**, quindi è adatto anche per ambienti con temperature molto basse fino a -20°C.

Inoltre, essendo classificato **IP67** (impermeabile e resistente alla polvere), offre una buona robustezza strutturale per ambienti difficili.

📄 Basato su 3 chunk


In [ ]:
# Test con domanda fuori dai documenti
domanda_off = "Quali sono i migliori smartphone del 2025?"
risposta_off, _ = chat_rag(domanda_off)
print(f"❓ {domanda_off}")
print(f"\n🤖 {risposta_off}")
print("\n💡 Il sistema dovrebbe rifiutarsi di rispondere!")

❓ Quali sono i migliori smartphone del 2025?

🤖 Non ho questa informazione nei miei documenti.

I documenti che ho a disposizione riguardano i prodotti e servizi di WiData Srl, un'azienda specializzata in IoT e smart cities. Non contengono informazioni su smartphone.

Posso invece aiutarti con domande su:
- Sensori IoT e gateway di WiData
- Piani tariffari e servizi cloud
- Supporto tecnico e contatti
- Certificazioni e specifiche tecniche dei prodotti

C'è qualcosa riguardante WiData di cui vorresti saperne di più?

💡 Il sistema dovrebbe rifiutarsi di rispondere!


---
## ⭐ Esercizi

In [ ]:
NOME_STUDENTE = "Lorenzo Masia"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Lorenzo Masia


### Esercizio 1 — Indicizza un documento tuo ★☆☆
Crea un documento di testo su un argomento a tua scelta (può essere anche una dispensa del corso, una ricetta, un regolamento). Indicizzalo in ChromaDB e fai 3 domande. I chunk recuperati sono rilevanti?

In [ ]:
# ESERCIZIO 1
import chromadb

chroma_client = chromadb.Client()

mio_documento = """
La regressione lineare è una tecnica di analisi dei dati che prevede il valore di dati sconosciuti utilizzando un altro valore di dati correlato e noto.

Modella matematicamente la variabile sconosciuta o dipendente e la variabile nota o indipendente come equazione lineare. Ad esempio, supponiamo di disporre di dati relativi alle spese e alle entrate dell'anno scorso. Le tecniche di regressione lineare analizzano questi dati e determinano che le tue spese sono la metà delle tue entrate. Quindi calcolano una spesa futura sconosciuta dimezzando un reddito noto futuro.

Fondamentalmente, una semplice tecnica di regressione lineare tenta di tracciare un grafico lineare tra due variabili di dati, x e y. Come variabile indipendente, x viene tracciata lungo l'asse orizzontale. Le variabili indipendenti sono anche chiamate variabili esplicative o variabili predittive.

La variabile dipendente, y, viene tracciata sull'asse verticale. È inoltre possibile fare riferimento ai valori y come variabili di risposta o variabili previste.
"""

# Crea una nuova collection
mia_collection = chroma_client.get_or_create_collection(name="mio_doc",
                                                        metadata={"hnsw:space": "cosine"})

def chunka_testo(testo, chunk_size=100, overlap=25):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = mia_collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Chunka e indicizza
chunks = chunka_testo(mio_documento)
mia_collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

domande= ['Cosa é la regressione lineare?',
          'Quali sono le variabili prese in considerazione?',
          'Cosa ci permette di fare la regressione?']

# Fai 3 domande e stampa i chunk recuperati
for domanda in domande:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk}...")
        print("-------------------------------------")
    print("\n")


❓ Cosa é la regressione lineare?
  📄 Chunk 1: 
La regressione lineare è una tecnica di analisi dei dati che prevede il valore di dati sconosciuti ...
-------------------------------------
  📄 Chunk 2: tecniche di regressione lineare analizzano questi dati e determinano che le tue spese sono la metà d...
-------------------------------------



❓ Quali sono le variabili prese in considerazione?
  📄 Chunk 1: ciata lungo l'asse orizzontale. Le variabili indipendenti sono anche chiamate variabili esplicative ...
-------------------------------------
  📄 Chunk 2: o ai valori y come variabili di risposta o variabili previste.
...
-------------------------------------



❓ Cosa ci permette di fare la regressione?
  📄 Chunk 1: 
La regressione lineare è una tecnica di analisi dei dati che prevede il valore di dati sconosciuti ...
-------------------------------------
  📄 Chunk 2: tecniche di regressione lineare analizzano questi dati e determinano che le tue spese sono la metà d...
-----------

### Esercizio 2 — Sperimenta con il chunking ★★☆
Prova a indicizzare lo stesso documento con chunk_size=200, 400 e 800. Per la stessa domanda, i chunk recuperati sono diversi? Quale dimensione dà risultati migliori?

In [ ]:
# ESERCIZIO 2
import chromadb

chroma_client = chromadb.Client()

def chunka_testo(testo, chunk_size=100, overlap=25):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = mia_collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

domanda_test = "Cosa é la regressione lineare?"  # ← modifica

for chunk_size in [200, 400, 800]:
    print(f"\n{'='*50}")
    print(f"chunk_size = {chunk_size}")
    print('='*50)


    # TODO: crea collection, chunka con dimensione diversa, indicizza, cerca
    collection = f"mio_doc_{chunk_size}"
    mia_collection = chroma_client.get_or_create_collection(name=collection,
                                                        metadata={"hnsw:space": "cosine"})

    chunks = chunka_testo(mio_documento, chunk_size=chunk_size)
    mia_collection.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )
    risultati = cerca(domanda_test)
    print(risultati[0])
    pass

# Commento: quale chunk_size ha dato i risultati migliori?
# Risposta: quella da 400 ha dato una risposta piú diretta, cosi come quella da 200, mentre quella da 800 fornisce forse qualche dettaglia in piú ma finisce col ripetersi nella spiegazi


chunk_size = 200
a sconosciuta dimezzando un reddito noto futuro.

Fondamentalmente, una semplice tecnica di regressione lineare tenta di tracciare un grafico lineare tra due variabili di dati, x e y. Come variabile i

chunk_size = 400
tecniche di regressione lineare analizzano questi dati e determinano che le tue spese sono la metà delle tue entrate. Quindi calcolano una spesa futura sconosciuta dimezzando un reddito noto futuro.

Fondamentalmente, una semplice tecnica di regressione lineare tenta di tracciare un grafico lineare tra due variabili di dati, x e y. Come variabile indipendente, x viene tracciata lungo l'asse orizzo

chunk_size = 800

La regressione lineare è una tecnica di analisi dei dati che prevede il valore di dati sconosciuti utilizzando un altro valore di dati correlato e noto.

Modella matematicamente la variabile sconosciuta o dipendente e la variabile nota o indipendente come equazione lineare. Ad esempio, supponiamo di disporre di dati relativi alle spese e all

### Esercizio 3 — RAG + storia conversazione ★★☆
Integra RAG nella funzione `chat()` della Lezione 3 (quella con la history). Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.

In [ ]:
# ESERCIZIO 3 - DEBUG MIGLIORATO
history = []

SYSTEM_PROMPT = """
Sei un coach esport esperto di Counter-Strike 2.
Rispondi usando SOLO i documenti forniti. Se una risposta non è presente, dichiara che l'informazione non è disponibile.
"""

def chunka_testo(testo, chunk_size=600, overlap=150):
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def chat(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history
    }
    if system: params["system"] = system
    risposta = client.messages.create(**params).content[0].text
    history.append({"role": "assistant", "content": risposta})
    return risposta

def cerca(domanda, n_risultati=3):
    risultati = collection.query(query_texts=[domanda], n_results=n_risultati)
    return risultati["documents"][0]

def chat_rag_con_storia(domanda):
    # 1. Recupero
    chunks_rilevanti = cerca(domanda, n_risultati=2)

    # 2. DEBUG MIGLIORATO: Mostra inizio e fine del chunk
    print(f"\n--- DEBUG: CHUNK RECUPERATI PER LA DOMANDA: '{domanda}' ---")
    for i, c in enumerate(chunks_rilevanti):
        testo_pulito = c.replace('\n', ' ').strip()
        # Mostriamo i primi 150 e gli ultimi 150 caratteri del chunk
        anteprima = f"{testo_pulito[:150]} [...] {testo_pulito[-150:]}"
        print(f"[Chunk {i+1}]: {anteprima}")
    print("-----------------------------------------------------------\n")

    # 3. Generazione
    contesto = "\n\n---\n\n".join(chunks_rilevanti)
    prompt = f"CONTESTO:\n{contesto}\n\n---\nDOMANDA: {domanda}"
    risposta = chat(prompt, system=SYSTEM_PROMPT)
    print(f"🤖 {risposta}")

documento_cs="""
Counter-Strike 2 (CS2) è uno sparatutto tattico competitivo sviluppato da Valve Corporation e pubblicato il 27 settembre 2023 come successore di Counter-Strike: Global Offensive. Il gioco utilizza il motore Source 2 e mantiene il classico formato 5 contro 5 tra Terrorist (T) e Counter-Terrorist (CT). Nella modalità Competitive le partite durano fino a un massimo di 24 round e vince la squadra che raggiunge 13 round; il rank è separato per ogni mappa. La modalità Premier rappresenta il sistema competitivo principale, utilizza il veto delle mappe e assegna un CS Rating numerico che può superare i 35.000 punti. Il map pool Premier considerato in questa documentazione comprende Mirage (de_mirage), Inferno (de_inferno), Dust II (de_dust2), Nuke (de_nuke), Ancient (de_ancient), Anubis (de_anubis) e Train (de_train). L'economia del gioco prevede un bonus sconfitta progressivo: 1.400$ dopo una sconfitta consecutiva, 1.900$ dopo due, 2.400$ dopo tre, 2.900$ dopo quattro e 3.400$ dopo cinque o più sconfitte consecutive. Le ricompense per eliminazione variano in base all'arma utilizzata: AWP 100$, AK-47 300$, M4A4 300$, M4A1-S 300$, Desert Eagle 300$, P90 600$, MP9 600$, MAC-10 600$ e XM1014 900$. L'AK-47 è disponibile ai Terrorist, costa 2.700$, infligge 36 danni base, possiede un caricatore da 30 colpi ed è in grado di eliminare con un singolo colpo alla testa un nemico dotato di casco. L'M4A4 è disponibile ai Counter-Terrorist, costa 3.000$, infligge 33 danni base e possiede un caricatore da 30 colpi. L'M4A1-S costa 2.900$, infligge 38 danni base, dispone di un caricatore da 20 colpi ed è equipaggiata con un silenziatore integrato. L'AWP costa 4.750$, utilizza un caricatore da 5 colpi ed elimina generalmente un avversario con un colpo al torso. Per quanto riguarda le protezioni, il Kevlar costa 650$ e riduce i danni ricevuti, mentre il Kevlar con casco costa 1.000$ e protegge anche dai colpi alla testa delle armi con penetrazione insufficiente. Tra le utility disponibili figurano la Flashbang dal costo di 200$ e con un effetto massimo di circa 2 secondi, la Smoke Grenade dal costo di 300$ e con una durata di circa 20 secondi che sfrutta il sistema di smoke volumetriche di Source 2, la HE Grenade dal costo di 300$ con danno variabile in base alla distanza, la Molotov dal costo di 400$ disponibile esclusivamente ai Terrorist e la Incendiary Grenade dal costo di 500$ disponibile esclusivamente ai Counter-Terrorist. Counter-Strike 2 utilizza un sistema definito sub-tick, nel quale il server continua a operare a tick regolari mentre le azioni del giocatore vengono registrate con una precisione temporale più elevata all'interno del tick stesso, con l'obiettivo di ridurre le discrepanze percepite tra input del giocatore e registrazione lato server. Tra i comandi console più utilizzati vi sono cl_showfps 1 per mostrare gli FPS, l'opzione di avvio -console per abilitare la console e cq_netgraph 1 per visualizzare la telemetria di rete. Il sistema di colori del CS Rating è suddiviso nelle seguenti fasce: da 0 a 4.999 Grigio, da 5.000 a 9.999 Azzurro, da 10.000 a 14.999 Blu, da 15.000 a 19.999 Viola, da 20.000 a 24.999 Rosa e oltre 25.000 Rosso.
La mappa de_santorini fa parte della rotazione competitiva avanzata, l'arma M4A5 è una variante dell'M4A4 con caricatore da 35 colpi e costo di 3.150$, il comando console cl_superfps 1 abilita una modalità di monitoraggio avanzata delle prestazioni e il rank Diamond Elite viene assegnato ai giocatori con un CS Rating superiore a 40.000 punti.
"""

# Reset database
chroma_client = chromadb.Client()
try: chroma_client.delete_collection("cs_debug_final")
except: pass

collection = chroma_client.create_collection(name="cs_debug_final")
chunks_debug = chunka_testo(documento_cs)
collection.add(documents=chunks_debug, ids=[f"df_{i}" for i in range(len(chunks_debug))])

# Test
chat_rag_con_storia("Quali sono i colori in base al rating?")


--- DEBUG: CHUNK RECUPERATI PER LA DOMANDA: 'Quali sono i colori in base al rating?' ---
[Chunk 1]: idurre le discrepanze percepite tra input del giocatore e registrazione lato server. Tra i comandi console più utilizzati vi sono cl_showfps 1 per mos [...] a 24.999 Rosa e oltre 25.000 Rosso. La mappa de_santorini fa parte della rotazione competitiva avanzata, l'arma M4A5 è una variante dell'M4A4 con cari
[Chunk 2]: sconfitta consecutiva, 1.900$ dopo due, 2.400$ dopo tre, 2.900$ dopo quattro e 3.400$ dopo cinque o più sconfitte consecutive. Le ricompense per elimi [...] singolo colpo alla testa un nemico dotato di casco. L'M4A4 è disponibile ai Counter-Terrorist, costa 3.000$, infligge 33 danni base e possiede un cari
-----------------------------------------------------------

🤖 Secondo i documenti forniti, il sistema di colori del CS Rating è suddiviso nelle seguenti fasce:

- **Da 0 a 4.999**: Grigio
- **Da 5.000 a 9.999**: Azzurro
- **Da 10.000 a 14.999**: Blu
- **Da 15.000 a 19.99

In [ ]:
chat_rag_con_storia("Esiste veramente l'm4a5?")


--- DEBUG: CHUNK RECUPERATI PER LA DOMANDA: 'Esiste veramente l'm4a5?' ---
[Chunk 1]: a 24.999 Rosa e oltre 25.000 Rosso. La mappa de_santorini fa parte della rotazione competitiva avanzata, l'arma M4A5 è una variante dell'M4A4 con cari [...] na modalità di monitoraggio avanzata delle prestazioni e il rank Diamond Elite viene assegnato ai giocatori con un CS Rating superiore a 40.000 punti.
[Chunk 2]: singolo colpo alla testa un nemico dotato di casco. L'M4A4 è disponibile ai Counter-Terrorist, costa 3.000$, infligge 33 danni base e possiede un cari [...] i, il Kevlar costa 650$ e riduce i danni ricevuti, mentre il Kevlar con casco costa 1.000$ e protegge anche dai colpi alla testa delle armi con penetr
-----------------------------------------------------------

🤖 Secondo i documenti forniti, sì, l'M4A5 esiste in Counter-Strike 2. È descritto come una variante dell'M4A4 con le seguenti caratteristiche:

- **Caricatore**: 35 colpi
- **Costo**: 3.150$

Tuttavia, devo precisare che ques

In [ ]:
chat_rag_con_storia("Esiste veramente la mappa de_santorini?")


--- DEBUG: CHUNK RECUPERATI PER LA DOMANDA: 'Esiste veramente la mappa de_santorini?' ---
[Chunk 1]: a 24.999 Rosa e oltre 25.000 Rosso. La mappa de_santorini fa parte della rotazione competitiva avanzata, l'arma M4A5 è una variante dell'M4A4 con cari [...] na modalità di monitoraggio avanzata delle prestazioni e il rank Diamond Elite viene assegnato ai giocatori con un CS Rating superiore a 40.000 punti.
[Chunk 2]: appa. La modalità Premier rappresenta il sistema competitivo principale, utilizza il veto delle mappe e assegna un CS Rating numerico che può superare [...] sconfitta consecutiva, 1.900$ dopo due, 2.400$ dopo tre, 2.900$ dopo quattro e 3.400$ dopo cinque o più sconfitte consecutive. Le ricompense per elimi
-----------------------------------------------------------

🤖 Secondo i documenti forniti, la mappa de_santorini viene menzionata come parte della rotazione competitiva avanzata.

Tuttavia, quando viene elencato il map pool Premier ufficiale, le mappe citate sono: Mirag

In [ ]:
chat_rag_con_storia("Nel gioco esiste veramente una rotazione competitiva avanzata?")


--- DEBUG: CHUNK RECUPERATI PER LA DOMANDA: 'Nel gioco esiste veramente una rotazione competitiva avanzata?' ---
[Chunk 1]: i, il Kevlar costa 650$ e riduce i danni ricevuti, mentre il Kevlar con casco costa 1.000$ e protegge anche dai colpi alla testa delle armi con penetr [...]  danno variabile in base alla distanza, la Molotov dal costo di 400$ disponibile esclusivamente ai Terrorist e la Incendiary Grenade dal costo di 500$
[Chunk 2]: appa. La modalità Premier rappresenta il sistema competitivo principale, utilizza il veto delle mappe e assegna un CS Rating numerico che può superare [...] sconfitta consecutiva, 1.900$ dopo due, 2.400$ dopo tre, 2.900$ dopo quattro e 3.400$ dopo cinque o più sconfitte consecutive. Le ricompense per elimi
-----------------------------------------------------------

🤖 Non posso rispondere a questa domanda basandomi SOLO sui documenti forniti, poiché essi non contengono informazioni specifiche su cosa sia o come sia strutturata una "rotazione competit

In [ ]:
chat_rag_con_storia("Ti ricordi se esiste veramente l'm4a5?")


--- DEBUG: CHUNK RECUPERATI PER LA DOMANDA: 'Ti ricordi se esiste veramente l'm4a5?' ---
[Chunk 1]: ata, l'arma M4A5 è una variante dell'M4A4 con caricatore da 35 colpi e costo di 3.150$, il comando console cl_superfps 1 abilita una modalità di monit [...] na modalità di monitoraggio avanzata delle prestazioni e il rank Diamond Elite viene assegnato ai giocatori con un CS Rating superiore a 40.000 punti.
[Chunk 2]: egistrate con una precisione temporale più elevata all'interno del tick stesso, con l'obiettivo di ridurre le discrepanze percepite tra input del gioc [...] ata, l'arma M4A5 è una variante dell'M4A4 con caricatore da 35 colpi e costo di 3.150$, il comando console cl_superfps 1 abilita una modalità di monit
-----------------------------------------------------------

🤖 In base ai documenti forniti, l'M4A5 è menzionato come "una variante dell'M4A4 con caricatore da 35 colpi e costo di 3.150$".

Tuttavia, devo precisare che, come coach esport di Counter-Strike 2, so che l'M4A5

In [ ]:
# TEST COMBINATO: RAG + HISTORY
history = []

print("--- STEP 1: RAG (Trova info su AK-47) ---")
chat_rag_con_storia("Parliamo dell'AK-47: quanto costa?")

print("\n--- STEP 2: HISTORY + RAG (Capisce 'questa' e cerca info relative) ---")
chat_rag_con_storia("Quanti danni infligge questa arma e quanti colpi ha?")

print("\n--- STEP 3: HISTORY + RAG (Confronto basato su memoria e documenti) ---")
chat_rag_con_storia("Costa più o meno dell'AWP?")

print("\n--- STEP 4: MEMORIA PURA (Verifica se ricorda il soggetto iniziale) ---")
chat_rag_con_storia("Qual era la prima arma di cui abbiamo parlato?")

--- STEP 1: RAG (Trova info su AK-47) ---

--- DEBUG: CHUNK RECUPERATI PER LA DOMANDA: 'Parliamo dell'AK-47: quanto costa?' ---
[Chunk 1]: n questa documentazione comprende Mirage (de_mirage), Inferno (de_inferno), Dust II (de_dust2), Nuke (de_nuke), Ancient (de_ancient), Anubis (de_anubi [...] ore da 30 colpi ed è in grado di eliminare con un singolo colpo alla testa un nemico dotato di casco. L'M4A4 è disponibile ai Counter-Terrorist, costa
[Chunk 2]: ata, l'arma M4A5 è una variante dell'M4A4 con caricatore da 35 colpi e costo di 3.150$, il comando console cl_superfps 1 abilita una modalità di monit [...] na modalità di monitoraggio avanzata delle prestazioni e il rank Diamond Elite viene assegnato ai giocatori con un CS Rating superiore a 40.000 punti.
-----------------------------------------------------------

🤖 # AK-47 - Costo

Secondo la documentazione fornita, l'**AK-47 costa 2.700$**.

Questa arma è disponibile per i Terrorist e infligge 36 danni base, con un caricatore da 30 

### Esercizio 4 — Chatbot RAG WiData completo ★★★ (Deliverable!)

Costruisci il chatbot completo con:
- RAG sul documento WiData
- Conversation history (sliding window)
- Streaming
- System prompt WiData con istruzione anti-hallucination
- Loop interattivo con `input()`
- Stampa i chunk usati per ogni risposta (per debug)

In [ ]:
# ESERCIZIO 4 — Chatbot RAG completo (DELIVERABLE)
import chromadb, json, os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

SYSTEM = """
Sei l'assistente virtuale ufficiale di WiData Srl, azienda IoT e smart cities con sede a Sassari.
Il tuo compito è aiutare gli utenti rispondendo esclusivamente sulla base dei "Documenti di riferimento" forniti di volta in volta.

Regole tassative anti-allucinazione:
1. Rispondi SOLO se l'informazione è esplicitamente presente nei documenti di riferimento forniti.
2. Se la risposta non si trova nei documenti, dì testualmente ed esclusivamente: 'Non ho questa informazione nei miei documenti.'
3. Non tentare di inventare, estrapolare o dedurre informazioni non scritte (es. prezzi non specificati o specifiche tecniche mancanti).
4. Sii conciso, professionale e preciso.
"""

MAX_MESSAGGI = 10

def setup_rag(testo):
    nome_coll = "widata_collection"
    collection = chroma_client.get_or_create_collection(
        name=nome_coll,
        metadata={"hnsw:space": "cosine"}
    )
    chunks = chunka_testo(testo)
    collection.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )

def cerca(domanda, collection, n=3):
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n)
    return risultati["documents"][0]

def chat_completo(domanda, history, collection):
    chunks_rilevanti = cerca(domanda, collection, n=2)
    contesto = "\n---\n".join(chunks_rilevanti)

    prompt_con_contesto = f"""Documenti di riferimento:
      {contesto}

      ---
      Domanda dell'utente: {domanda}"""

    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]


    messaggi_da_inviare = list(history)
    messaggi_da_inviare.append({"role": "user", "content": prompt_con_contesto})

    print("\n🤖 Assistente: ", end="", flush=True)


    testo_risposta = ""
    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        system=SYSTEM,
        messages=messaggi_da_inviare
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            testo_risposta += text
    print("\n")

    history.append({"role": "user", "content": domanda})
    history.append({"role": "assistant", "content": testo_risposta})

documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

def main():
    collection = setup_rag(documento_widata)
    history = []
    print("🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        chat_completo(utente, history, collection)

main()  # Decommentare per eseguire

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione4_TUONOME.ipynb`
4. Carica su GitHub in `lezione4/`

```bash
git add lezione4/
git commit -m "Lezione 4 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 04/06)
Leggi **Huyen Cap. 6 — sezione Agents**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*